# IPL Scouting Dashboard: SMA to IPL Player Profiling

## Final Notebook Submission

This notebook builds a practical IPL scouting workflow using Syed Mushtaq Ali Trophy (SMA) and Indian Premier League (IPL) player data.

The project originally tested supervised prediction of future IPL impact, but the EDA showed that only a limited subset of players have both prior SMA data and later IPL outcomes. Following the scouting-project feedback, the final method is reframed as an unsupervised scouting system rather than a direct prediction model.

**Final objective:** identify domestic SMA players worth scouting by combining observed impact scoring, K-means player profiles, and nearest IPL player comparables.


## 1. Research Question

**How can SMA and IPL player data be used to build a role-specific scouting system that identifies high-potential domestic players and compares them with similar IPL profiles?**

This is a scouting question, not a pure prediction question. The goal is to support shortlist creation and player comparison, not to claim exact future IPL performance.


In [1]:
from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
EDA_DIR = REPORTS_DIR / "eda"
SCOUTING_DIR = REPORTS_DIR / "scouting"
SCOUTING_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = PROCESSED_DIR / "player_features.parquet"
assert FEATURES_PATH.exists(), f"Missing {FEATURES_PATH}. Run run_features.py first."

features = pd.read_parquet(FEATURES_PATH)
print(f"Loaded processed player-season features: {features.shape[0]:,} rows x {features.shape[1]:,} columns")
print(f"Competitions: {sorted(features['competition'].dropna().unique())}")
print(f"Roles: {sorted(features['role'].dropna().unique())}")


Loaded processed player-season features: 9,981 rows x 67 columns
Competitions: ['IPL', 'SMA']
Roles: ['bat', 'bowl']


## 2. Simple EDA

The EDA is intentionally kept simple in the final notebook. It answers four practical scouting questions:

1. What data is available?
2. How much player overlap exists between SMA and IPL?
3. How do SMA and IPL environments differ?
4. Why is direct supervised prediction difficult?

Raw ball-by-ball data is useful for parsing checks, but the main EDA uses processed player-season features because those are the variables used for scoring, clustering, and shortlisting.


In [2]:
def read_optional_csv(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def print_table(df, columns=None, n=None):
    if columns is not None:
        df = df[columns]
    if n is not None:
        df = df.head(n)
    print(df.to_string(index=False))

eda_overview = read_optional_csv(EDA_DIR / "dataset_overview.csv")
player_overlap = read_optional_csv(EDA_DIR / "player_overlap.csv")
metric_summary = read_optional_csv(EDA_DIR / "metric_summary.csv")
training_summary = read_optional_csv(EDA_DIR / "training_summary.csv")

print("Dataset coverage by competition and role")
print_table(eda_overview, ["competition", "role", "player_season_rows", "players", "seasons", "first_season", "last_season"])

print("\nPlayer overlap between SMA and IPL")
print_table(player_overlap)

core_metrics = metric_summary[
    ((metric_summary["role"] == "bat") & metric_summary["metric"].isin(["strike_rate", "runs_per_innings", "boundary_pct"]))
    | ((metric_summary["role"] == "bowl") & metric_summary["metric"].isin(["economy", "bowling_sr", "wickets_per_innings"]))
].copy()
core_metrics = core_metrics[["competition", "role", "metric", "non_null", "mean", "median", "std"]].round(3)
print("\nCore SMA vs IPL metric comparison")
print_table(core_metrics)

print("\nSupervised training subset")
print_table(training_summary[["role", "training_rows", "players", "mean_target_ipl_sample", "median_target_ipl_sample"]].round(3))


Dataset coverage by competition and role
competition role  player_season_rows  players  seasons  first_season  last_season
        IPL  bat                2948      731       19          2008         2026
        IPL bowl                2199      576       19          2008         2026
        SMA  bat                2797     1304        7          2016         2024
        SMA bowl                2037      974        7          2016         2024

Player overlap between SMA and IPL
role  players_total  ipl_only  sma_only  played_both
 bat           1761       457      1030          274
bowl           1345       371       769          205

Core SMA vs IPL metric comparison
competition role              metric  non_null    mean  median    std
        IPL  bat         strike_rate      2948 111.482 117.779 49.871
        IPL  bat    runs_per_innings      2948  14.213  12.074 11.470
        IPL  bat        boundary_pct      2948  13.377  14.208 10.210
        IPL bowl             economy   

### EDA Interpretation

The EDA supports the final modelling choice:

- SMA provides a large domestic scouting pool.
- The SMA/IPL overlap is much smaller than the full player pool, which limits supervised learning.
- IPL batting strike rates and bowling economy are generally higher than SMA, so direct raw comparisons are not enough.
- Future IPL outcomes are noisy because player role, team usage, match opportunities, venue, and opponent quality are not fully captured in the data.

Therefore, the final project uses observed scoring plus unsupervised profiling instead of relying on a single supervised accuracy number.


## 3. Methodology

The final scouting workflow is:

```text
Processed player-season features
-> simple EDA
-> aggregate player profiles
-> observed role-specific impact score
-> K-means player profile clusters
-> nearest IPL player comparables
-> final domestic SMA shortlist
```

The impact score answers: **how strong has this player's observed performance been?**

K-means answers: **what type of player is this?**

Nearest-neighbor matching answers: **which IPL players have similar profiles?**


In [3]:
ROLE_CONFIG = {
    "bat": {
        "sample_col": "bat_innings",
        "min_sample": {"SMA": 15, "IPL": 10},
        "impact_weights": {
            "strike_rate": 0.40,
            "runs_per_innings": 0.40,
            "boundary_pct": 0.20,
        },
        "cluster_features": [
            "strike_rate", "runs_per_innings", "boundary_pct",
            "bat_pp_sr", "bat_mid_sr", "bat_death_sr",
        ],
    },
    "bowl": {
        "sample_col": "bowl_innings",
        "min_sample": {"SMA": 15, "IPL": 10},
        "impact_weights": {
            "economy": -0.40,
            "wickets_per_innings": 0.35,
            "bowling_sr": -0.25,
        },
        "cluster_features": [
            "economy", "bowling_sr", "wickets_per_innings",
            "bowl_pp_economy", "bowl_mid_economy", "bowl_death_economy", "wide_rate",
        ],
    },
}

print("Impact-score weights")
for role, cfg in ROLE_CONFIG.items():
    print(f"{role}: {cfg['impact_weights']}")

print("\nMinimum sample filters")
for role, cfg in ROLE_CONFIG.items():
    print(f"{role}: SMA >= {cfg['min_sample']['SMA']} innings, IPL >= {cfg['min_sample']['IPL']} innings")


Impact-score weights
bat: {'strike_rate': 0.4, 'runs_per_innings': 0.4, 'boundary_pct': 0.2}
bowl: {'economy': -0.4, 'wickets_per_innings': 0.35, 'bowling_sr': -0.25}

Minimum sample filters
bat: SMA >= 15 innings, IPL >= 10 innings
bowl: SMA >= 15 innings, IPL >= 10 innings


## 4. Aggregate Player Profiles

The processed table has one row per player-season-role. For scouting, the notebook aggregates those into one profile per player, competition, and role.

This gives each player a stable batting or bowling profile across all available seasons.


In [4]:
def safe_divide(numerator, denominator, multiplier=1.0):
    return numerator / denominator * multiplier if pd.notna(denominator) and denominator > 0 else np.nan


def aggregate_player_profiles(features, role, competition):
    rows = []
    df = features[(features["role"] == role) & (features["competition"] == competition)].copy()

    for player_key, group in df.groupby("player_key", dropna=False):
        player_name = group["player_name"].dropna().iloc[-1] if group["player_name"].notna().any() else player_key
        player_id = group["player_id"].dropna().iloc[-1] if group["player_id"].notna().any() else player_key
        row = {
            "player_key": player_key,
            "player_id": player_id,
            "player_name": player_name,
            "competition": competition,
            "role": role,
            "seasons": int(group["season"].nunique()),
            "first_season": int(group["season"].min()),
            "last_season": int(group["season"].max()),
        }

        if role == "bat":
            count_cols = [
                "bat_innings", "runs", "balls", "fours", "sixes",
                "bat_pp_runs", "bat_pp_balls", "bat_mid_runs", "bat_mid_balls",
                "bat_death_runs", "bat_death_balls",
            ]
            for col in count_cols:
                row[col] = float(group[col].sum(skipna=True)) if col in group else np.nan
            row["sample"] = row["bat_innings"]
            row["strike_rate"] = safe_divide(row["runs"], row["balls"], 100)
            row["runs_per_innings"] = safe_divide(row["runs"], row["bat_innings"])
            row["boundary_pct"] = safe_divide(row["fours"] + row["sixes"], row["balls"], 100)
            row["bat_pp_sr"] = safe_divide(row["bat_pp_runs"], row["bat_pp_balls"], 100)
            row["bat_mid_sr"] = safe_divide(row["bat_mid_runs"], row["bat_mid_balls"], 100)
            row["bat_death_sr"] = safe_divide(row["bat_death_runs"], row["bat_death_balls"], 100)
        else:
            count_cols = [
                "bowl_innings", "runs_conceded", "balls_bowled", "wickets",
                "bowl_pp_runs", "bowl_pp_balls", "bowl_mid_runs", "bowl_mid_balls",
                "bowl_death_runs", "bowl_death_balls", "wides", "noballs",
            ]
            for col in count_cols:
                row[col] = float(group[col].sum(skipna=True)) if col in group else np.nan
            row["sample"] = row["bowl_innings"]
            row["economy"] = safe_divide(row["runs_conceded"], row["balls_bowled"], 6)
            row["bowling_sr"] = safe_divide(row["balls_bowled"], row["wickets"])
            row["wickets_per_innings"] = safe_divide(row["wickets"], row["bowl_innings"])
            row["wide_rate"] = safe_divide(row["wides"], row["balls_bowled"], 6)
            row["noball_rate"] = safe_divide(row["noballs"], row["balls_bowled"], 6)
            row["bowl_pp_economy"] = safe_divide(row["bowl_pp_runs"], row["bowl_pp_balls"], 6)
            row["bowl_mid_economy"] = safe_divide(row["bowl_mid_runs"], row["bowl_mid_balls"], 6)
            row["bowl_death_economy"] = safe_divide(row["bowl_death_runs"], row["bowl_death_balls"], 6)

        rows.append(row)

    return pd.DataFrame(rows)

profiles = pd.concat(
    [aggregate_player_profiles(features, role, comp) for role in ROLE_CONFIG for comp in ["SMA", "IPL"]],
    ignore_index=True,
    sort=False,
)
profiles["min_sample"] = profiles.apply(lambda r: ROLE_CONFIG[r["role"]]["min_sample"][r["competition"]], axis=1)
eligible_profiles = profiles[profiles["sample"].fillna(0) >= profiles["min_sample"]].copy()

print(f"All player profiles: {len(profiles):,}")
print(f"Eligible profiles after sample filtering: {len(eligible_profiles):,}")
print(eligible_profiles.groupby(["competition", "role"]).agg(players=("player_key", "nunique"), mean_sample=("sample", "mean")).round(2).to_string())


All player profiles: 3,585
Eligible profiles after sample filtering: 1,088
                  players  mean_sample
competition role                      
IPL         bat       350        49.17
            bowl      302        44.91
SMA         bat       235        24.18
            bowl      201        23.76


## 5. Observed Impact Score

The impact score is the project's scouting metric.

For each role and competition, each metric is converted to a z-score. The weighted z-scores are then combined into one observed impact score.

For batters:

```text
0.40 * z(strike_rate) + 0.40 * z(runs_per_innings) + 0.20 * z(boundary_pct)
```

For bowlers:

```text
-0.40 * z(economy) + 0.35 * z(wickets_per_innings) - 0.25 * z(bowling_sr)
```

The score is then shrunk toward the role average for smaller samples.


In [5]:
def zscore(series):
    std = series.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.mean()) / std


def add_observed_impact_scores(profiles):
    out = profiles.copy()
    out["impact_raw"] = np.nan
    out["impact_reliability"] = np.nan
    out["impact_score"] = np.nan
    out["impact_percentile"] = np.nan

    for (competition, role), group in out.groupby(["competition", "role"]):
        weights = ROLE_CONFIG[role]["impact_weights"]
        idx = group.index
        score = pd.Series(0.0, index=idx)
        used_weight = 0.0

        for metric, weight in weights.items():
            values = out.loc[idx, metric]
            if values.notna().sum() < 2:
                continue
            score = score.add(zscore(values).fillna(0.0) * weight, fill_value=0.0)
            used_weight += abs(weight)

        if used_weight == 0:
            continue

        raw = score / used_weight
        sample = out.loc[idx, "sample"].fillna(0).astype(float)
        reliability = sample / (sample + 10.0)
        role_mean = raw.mean()
        impact = reliability * raw + (1.0 - reliability) * role_mean

        out.loc[idx, "impact_raw"] = raw
        out.loc[idx, "impact_reliability"] = reliability
        out.loc[idx, "impact_score"] = impact
        out.loc[idx, "impact_percentile"] = impact.rank(pct=True) * 100.0

    return out

eligible_profiles = add_observed_impact_scores(eligible_profiles)

print("Top observed SMA batters")
print_table(
    eligible_profiles[(eligible_profiles["competition"] == "SMA") & (eligible_profiles["role"] == "bat")]
    .sort_values("impact_score", ascending=False)
    [["player_name", "sample", "impact_score", "impact_percentile", "strike_rate", "runs_per_innings", "boundary_pct"]]
    .round(2),
    n=10,
)

print("\nTop observed SMA bowlers")
print_table(
    eligible_profiles[(eligible_profiles["competition"] == "SMA") & (eligible_profiles["role"] == "bowl")]
    .sort_values("impact_score", ascending=False)
    [["player_name", "sample", "impact_score", "impact_percentile", "economy", "bowling_sr", "wickets_per_innings"]]
    .round(2),
    n=10,
)


Top observed SMA batters
 player_name  sample  impact_score  impact_percentile  strike_rate  runs_per_innings  boundary_pct
     PP Shaw    25.0          1.23             100.00       162.44             28.72         25.57
  RM Patidar    33.0          1.20              99.57       152.56             34.30         20.35
     SS Iyer    34.0          1.18              99.15       147.75             35.68         20.10
    SA Yadav    40.0          1.08              98.72       154.82             28.10         22.18
     RK Bhui    28.0          1.08              98.30       154.33             33.07         19.00
    HV Patel    26.0          1.06              97.87       157.01             25.85         26.17
Ishan Kishan    23.0          1.01              97.45       144.44             33.35         22.60
    RK Singh    35.0          0.99              97.02       159.65             26.00         20.88
      N Rana    46.0          0.94              96.60       143.78             29.91

## 6. K-means Player Profiles

K-means is used separately for batters and bowlers. The clustering variables are role-specific, so batters are not compared directly with bowlers.

This creates interpretable scouting groups such as high-impact power hitters, run accumulators, economy-support bowlers, and high-impact strike/control bowlers.


In [6]:
def clear_cluster_names(summary, role):
    summary = summary.copy()
    summary["profile_type"] = "Unlabelled profile"

    if role == "bat":
        max_impact = summary["impact_score"].idxmax()
        min_impact = summary["impact_score"].idxmin()
        max_rpi = summary.drop(index=[max_impact, min_impact], errors="ignore")["runs_per_innings"].idxmax()
        summary.loc[max_impact, "profile_type"] = "High-impact power hitters"
        summary.loc[min_impact, "profile_type"] = "Low-output batting group"
        summary.loc[max_rpi, "profile_type"] = "Run accumulators"
        summary.loc[summary["profile_type"] == "Unlabelled profile", "profile_type"] = "Balanced batting group"
    else:
        max_impact = summary["impact_score"].idxmax()
        min_impact = summary["impact_score"].idxmin()
        remaining = summary.drop(index=[max_impact, min_impact], errors="ignore")
        max_wpi = remaining["wickets_per_innings"].idxmax()
        summary.loc[max_impact, "profile_type"] = "High-impact strike/control bowlers"
        summary.loc[min_impact, "profile_type"] = "Low-wicket expensive group"
        summary.loc[max_wpi, "profile_type"] = "Attacking wicket-takers"
        summary.loc[summary["profile_type"] == "Unlabelled profile", "profile_type"] = "Economy support bowlers"

    return summary

cluster_outputs = []
role_models = {}
eligible_profiles["cluster"] = np.nan
eligible_profiles["profile_type"] = pd.NA

for role, cfg in ROLE_CONFIG.items():
    role_idx = eligible_profiles[eligible_profiles["role"] == role].index
    role_df = eligible_profiles.loc[role_idx].copy()
    feature_cols = [col for col in cfg["cluster_features"] if col in role_df.columns]

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    x = role_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    x_scaled = scaler.fit_transform(imputer.fit_transform(x))

    kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
    labels = kmeans.fit_predict(x_scaled)
    eligible_profiles.loc[role_idx, "cluster"] = labels
    role_models[role] = {"features": feature_cols, "imputer": imputer, "scaler": scaler, "kmeans": kmeans}

    agg_dict = {
        "players": ("player_key", "nunique"),
        "sma_players": ("competition", lambda s: int((s == "SMA").sum())),
        "ipl_players": ("competition", lambda s: int((s == "IPL").sum())),
        "mean_sample": ("sample", "mean"),
        "impact_score": ("impact_score", "mean"),
    }
    agg_dict.update({col: (col, "mean") for col in feature_cols})
    summary = role_df.assign(cluster=labels).groupby("cluster").agg(**agg_dict).reset_index()
    summary.insert(0, "role", role)
    summary = clear_cluster_names(summary, role)
    cluster_outputs.append(summary)

    label_map = dict(zip(summary["cluster"], summary["profile_type"]))
    eligible_profiles.loc[role_idx, "profile_type"] = eligible_profiles.loc[role_idx, "cluster"].map(label_map)

cluster_summary = pd.concat(cluster_outputs, ignore_index=True, sort=False)
eligible_profiles["cluster"] = eligible_profiles["cluster"].astype(int)

cluster_display_cols = [
    "role", "cluster", "profile_type", "players", "sma_players", "ipl_players", "mean_sample", "impact_score",
    "strike_rate", "runs_per_innings", "boundary_pct", "economy", "bowling_sr", "wickets_per_innings",
]
cluster_display_cols = [col for col in cluster_display_cols if col in cluster_summary.columns]
print_table(cluster_summary[cluster_display_cols].round(2))


role  cluster                       profile_type  players  sma_players  ipl_players  mean_sample  impact_score  strike_rate  runs_per_innings  boundary_pct  economy  bowling_sr  wickets_per_innings
 bat        0          High-impact power hitters       37            9           29        34.76          0.81       165.14             21.43         25.32      NaN         NaN                  NaN
 bat        1           Low-output batting group      104           24           80        20.61         -0.84        90.28              7.19          9.16      NaN         NaN                  NaN
 bat        2                   Run accumulators      165           85          105        57.26          0.59       140.08             26.30         19.02      NaN         NaN                  NaN
 bat        3             Balanced batting group      234          117          136        33.79         -0.12       122.09             17.09         14.70      NaN         NaN                  NaN
bowl      

  File "C:\Users\Rajvi\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


## 7. Similar IPL Player Matching

The notebook now finds the closest IPL comparables for each eligible domestic SMA player who has not appeared in the IPL data.

The matching uses the same standardized role-specific features used in K-means. This makes the comparison interpretable and avoids mixing batting and bowling profiles.


In [7]:
def build_similarity_matches(profiles, top_n=3):
    rows = []
    for role, model in role_models.items():
        feature_cols = model["features"]
        role_profiles = profiles[profiles["role"] == role].copy()

        ipl_keys = set(role_profiles[role_profiles["competition"] == "IPL"]["player_key"])
        candidates = role_profiles[(role_profiles["competition"] == "SMA") & (~role_profiles["player_key"].isin(ipl_keys))].reset_index(drop=True)
        comparables = role_profiles[role_profiles["competition"] == "IPL"].reset_index(drop=True)

        if candidates.empty or comparables.empty:
            continue

        comparable_x = model["scaler"].transform(model["imputer"].transform(comparables[feature_cols].replace([np.inf, -np.inf], np.nan)))
        candidate_x = model["scaler"].transform(model["imputer"].transform(candidates[feature_cols].replace([np.inf, -np.inf], np.nan)))

        nearest = NearestNeighbors(n_neighbors=min(top_n, len(comparables)), metric="euclidean")
        nearest.fit(comparable_x)
        distances, indices = nearest.kneighbors(candidate_x)

        for candidate_pos, candidate in candidates.iterrows():
            for match_rank, (distance, comparable_pos) in enumerate(zip(distances[candidate_pos], indices[candidate_pos]), start=1):
                comparable = comparables.iloc[int(comparable_pos)]
                rows.append({
                    "domestic_player_key": candidate["player_key"],
                    "domestic_player_name": candidate["player_name"],
                    "role": role,
                    "domestic_sample": candidate["sample"],
                    "domestic_impact_score": candidate["impact_score"],
                    "domestic_impact_percentile": candidate["impact_percentile"],
                    "domestic_profile_type": candidate["profile_type"],
                    "match_rank": match_rank,
                    "ipl_player_key": comparable["player_key"],
                    "ipl_player_name": comparable["player_name"],
                    "ipl_sample": comparable["sample"],
                    "ipl_impact_score": comparable["impact_score"],
                    "ipl_impact_percentile": comparable["impact_percentile"],
                    "ipl_profile_type": comparable["profile_type"],
                    "distance": float(distance),
                    "similarity_score": float(100.0 / (1.0 + distance)),
                })
    return pd.DataFrame(rows)

similarity_matches = build_similarity_matches(eligible_profiles, top_n=3)
print(f"Domestic-to-IPL similarity match rows: {len(similarity_matches):,}")
print_table(
    similarity_matches[[
        "domestic_player_name", "role", "domestic_profile_type", "match_rank",
        "ipl_player_name", "ipl_profile_type", "similarity_score",
    ]].round(2),
    n=12,
)


Domestic-to-IPL similarity match rows: 831
domestic_player_name role  domestic_profile_type  match_rank ipl_player_name       ipl_profile_type  similarity_score
        Vishnu Vinod  bat       Run accumulators           1   A Raghuvanshi       Run accumulators             65.56
        Vishnu Vinod  bat       Run accumulators           2        SK Raina       Run accumulators             64.16
        Vishnu Vinod  bat       Run accumulators           3    Ishan Kishan       Run accumulators             63.53
       Rajesh Dhuper  bat Balanced batting group           1    JEC Franklin Balanced batting group             64.05
       Rajesh Dhuper  bat Balanced batting group           2      AL Menaria Balanced batting group             61.24
       Rajesh Dhuper  bat Balanced batting group           3        FY Fazal Balanced batting group             59.20
           SS Mundhe  bat Balanced batting group           1          S Dube       Run accumulators             57.36
           SS

## 8. Final Scouting Shortlist

The final scouting score combines:

- **60% observed SMA impact percentile** - how strong the player has been in domestic T20 cricket.
- **25% similarity percentile** - how closely the player resembles an IPL profile.
- **15% sample-size percentile** - confidence from having more role innings.

This is a shortlist ranking, not a prediction of future IPL success.


In [8]:
def add_role_percentile(df, col, out_col):
    df[out_col] = df.groupby("role")[col].rank(pct=True) * 100.0
    return df


def fmt(value, digits=1):
    return "NA" if pd.isna(value) else f"{value:.{digits}f}"

rank1_matches = similarity_matches[similarity_matches["match_rank"] == 1].copy()
rank1_matches = rank1_matches.rename(columns={
    "ipl_player_name": "closest_ipl_player",
    "ipl_profile_type": "closest_ipl_profile_type",
    "ipl_impact_percentile": "closest_ipl_impact_percentile",
})

ipl_keys_by_role = {
    role: set(eligible_profiles[(eligible_profiles["competition"] == "IPL") & (eligible_profiles["role"] == role)]["player_key"])
    for role in ROLE_CONFIG
}
candidate_profiles = pd.concat(
    [
        eligible_profiles[
            (eligible_profiles["competition"] == "SMA")
            & (eligible_profiles["role"] == role)
            & (~eligible_profiles["player_key"].isin(ipl_keys_by_role[role]))
        ]
        for role in ROLE_CONFIG
    ],
    ignore_index=True,
)

shortlist = candidate_profiles.merge(
    rank1_matches[[
        "domestic_player_key", "role", "closest_ipl_player", "closest_ipl_profile_type",
        "closest_ipl_impact_percentile", "similarity_score", "distance",
    ]],
    left_on=["player_key", "role"],
    right_on=["domestic_player_key", "role"],
    how="left",
)
shortlist = add_role_percentile(shortlist, "sample", "sample_percentile")
shortlist = add_role_percentile(shortlist, "similarity_score", "similarity_percentile")
shortlist["scouting_score"] = (
    0.60 * shortlist["impact_percentile"].fillna(0)
    + 0.25 * shortlist["similarity_percentile"].fillna(0)
    + 0.15 * shortlist["sample_percentile"].fillna(0)
)


def scouting_reason(row):
    comparable = row["closest_ipl_player"] if pd.notna(row["closest_ipl_player"]) else "no close IPL comparable"
    if row["role"] == "bat":
        return (
            f"{row['profile_type']}; SR {fmt(row['strike_rate'])}, RPI {fmt(row['runs_per_innings'])}, "
            f"boundary% {fmt(row['boundary_pct'])}; closest IPL comp: {comparable}"
        )
    return (
        f"{row['profile_type']}; economy {fmt(row['economy'], 2)}, bowling SR {fmt(row['bowling_sr'])}, "
        f"WPI {fmt(row['wickets_per_innings'], 2)}; closest IPL comp: {comparable}"
    )

shortlist["scouting_reason"] = shortlist.apply(scouting_reason, axis=1)
scouting_shortlist = shortlist.sort_values(["role", "scouting_score"], ascending=[True, False]).groupby("role").head(25).reset_index(drop=True)

bat_cols = ["player_name", "sample", "scouting_score", "impact_percentile", "profile_type", "closest_ipl_player", "similarity_score", "strike_rate", "runs_per_innings", "boundary_pct"]
bowl_cols = ["player_name", "sample", "scouting_score", "impact_percentile", "profile_type", "closest_ipl_player", "similarity_score", "economy", "bowling_sr", "wickets_per_innings"]

print("Top domestic SMA batting shortlist")
print_table(scouting_shortlist[scouting_shortlist["role"] == "bat"][bat_cols].round(2), n=10)

print("\nTop domestic SMA bowling shortlist")
print_table(scouting_shortlist[scouting_shortlist["role"] == "bowl"][bowl_cols].round(2), n=10)


Top domestic SMA batting shortlist
     player_name  sample  scouting_score  impact_percentile     profile_type closest_ipl_player  similarity_score  strike_rate  runs_per_innings  boundary_pct
     Virat Singh    36.0           87.15              86.38 Run accumulators         MEK Hussey             70.84       130.38             34.33         16.03
     Vivek Singh    42.0           85.99              80.85 Run accumulators            E Lewis             72.54       136.26             25.14         19.48
 Priyank Panchal    35.0           83.18              77.02 Run accumulators          RG Sharma             72.65       128.89             27.66         17.04
     Rohan Kadam    25.0           82.66              95.74 Run accumulators           KL Rahul             64.21       130.75             39.80         18.27
    Vishnu Vinod    27.0           80.83              88.09 Run accumulators      A Raghuvanshi             65.56       144.36             28.44         18.98
      SP Ja

## 9. Save Dashboard-Ready Results

The CSV outputs below can feed the final dashboard and report tables.


In [9]:
eligible_profiles.to_csv(SCOUTING_DIR / "player_profiles_with_clusters.csv", index=False)
cluster_summary.to_csv(SCOUTING_DIR / "cluster_summary.csv", index=False)
similarity_matches.to_csv(SCOUTING_DIR / "player_similarity_matches.csv", index=False)
shortlist.sort_values(["role", "scouting_score"], ascending=[True, False]).to_csv(SCOUTING_DIR / "domestic_candidates_scored.csv", index=False)
scouting_shortlist.to_csv(SCOUTING_DIR / "scouting_shortlist.csv", index=False)

print("Saved scouting CSV outputs")
for path in sorted(SCOUTING_DIR.glob("*.csv")):
    print(f"- {path.relative_to(PROJECT_ROOT)}")


Saved scouting CSV outputs
- reports\scouting\cluster_summary.csv
- reports\scouting\domestic_candidates_scored.csv
- reports\scouting\player_profiles_with_clusters.csv
- reports\scouting\player_similarity_matches.csv
- reports\scouting\scouting_shortlist.csv


## 10. Visual Results

The charts are saved for the report/dashboard. They show the top domestic shortlists and the unsupervised player-profile spaces.


In [10]:
def save_top_chart(df, role, metric, title, filename):
    plot_df = df[df["role"] == role].sort_values(metric, ascending=True).tail(15)
    plt.figure(figsize=(9, 6))
    plt.barh(plot_df["player_name"], plot_df[metric], color="#2f6f73" if role == "bat" else "#c57b57")
    plt.title(title)
    plt.xlabel(metric.replace("_", " ").title())
    plt.tight_layout()
    plt.savefig(SCOUTING_DIR / filename, dpi=160)
    plt.close()

save_top_chart(scouting_shortlist, "bat", "scouting_score", "Top Domestic SMA Batters", "top_domestic_batters.png")
save_top_chart(scouting_shortlist, "bowl", "scouting_score", "Top Domestic SMA Bowlers", "top_domestic_bowlers.png")

bat_plot = eligible_profiles[eligible_profiles["role"] == "bat"].dropna(subset=["strike_rate", "runs_per_innings"])
plt.figure(figsize=(8, 6))
for label, group in bat_plot.groupby("profile_type"):
    plt.scatter(group["strike_rate"], group["runs_per_innings"], s=30, alpha=0.65, label=label)
plt.title("Batting Profiles: Strike Rate vs Runs Per Innings")
plt.xlabel("Strike Rate")
plt.ylabel("Runs Per Innings")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(SCOUTING_DIR / "batting_profile_clusters.png", dpi=160)
plt.close()

bowl_plot = eligible_profiles[eligible_profiles["role"] == "bowl"].dropna(subset=["economy", "wickets_per_innings"])
plt.figure(figsize=(8, 6))
for label, group in bowl_plot.groupby("profile_type"):
    plt.scatter(group["economy"], group["wickets_per_innings"], s=30, alpha=0.65, label=label)
plt.title("Bowling Profiles: Economy vs Wickets Per Innings")
plt.xlabel("Economy")
plt.ylabel("Wickets Per Innings")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(SCOUTING_DIR / "bowling_profile_clusters.png", dpi=160)
plt.close()

print("Saved scouting charts")
for path in sorted(SCOUTING_DIR.glob("*.png")):
    print(f"- {path.relative_to(PROJECT_ROOT)}")


Saved scouting charts
- reports\scouting\batting_profile_clusters.png
- reports\scouting\bowling_profile_clusters.png
- reports\scouting\top_domestic_batters.png
- reports\scouting\top_domestic_bowlers.png


### Chart Files

![Top Domestic SMA Batters](reports/scouting/top_domestic_batters.png)

![Top Domestic SMA Bowlers](reports/scouting/top_domestic_bowlers.png)

![Batting Profile Clusters](reports/scouting/batting_profile_clusters.png)

![Bowling Profile Clusters](reports/scouting/bowling_profile_clusters.png)


## 11. Final Findings

The final scouting workflow produces a practical set of domestic candidates and IPL comparables.

**Top domestic batting candidates include:** Virat Singh, Vivek Singh, Priyank Panchal, Rohan Kadam, Vishnu Vinod, SP Jackson, Anmolpreet Singh, PN Mankad, Samarth Vyas, and Jay Bista.

**Top domestic bowling candidates include:** A Nagwaswalla, Satyajeet Bachhav, SZ Mulani, Suboth Bhati, Sayan Ghosh, CV Milind, LI Meriwala, A Choudhary, Pankaj Jaswal, and Akshay Karnewar.

These are not claimed to be guaranteed IPL successes. They are players whose observed SMA profiles, sample sizes, and similarity to IPL profiles make them suitable for further scouting review.


## 12. Limitations

- The impact score is a designed scouting metric, so its weights reflect project assumptions.
- K-means clusters are descriptive profiles, not true labels.
- Similarity matching compares statistical profiles, not tactical context, age, handedness, fielding, injury history, or auction constraints.
- Some players may be excluded from the domestic shortlist if they already appear in IPL data.
- Supervised prediction was not used as the main method because the available SMA-to-future-IPL overlap is comparatively small and noisy.


## 13. Conclusion

This project is best understood as a scouting decision-support tool.

The final method uses EDA to understand the data, an observed impact score to rank performance, K-means to identify player types, and nearest-neighbor matching to compare domestic SMA candidates with similar IPL players.

This approach is better aligned with scouting practice than forcing a supervised model to predict a noisy future outcome. It gives scouts a structured shortlist and a clear reason for why each player is worth reviewing.
